# 01 - How Large Language Models Work

**Course:** FDP on Agentic AI & LLM Engineering

**Environment**
- Python 3.12
- VS Code
- Ollama
- Model: qwen3:4b
- API: OpenAI Compatible API (Local)

---

## Objective

This notebook explores the fundamentals of Large Language Models (LLMs) and demonstrates how to interact with a locally hosted LLM using Ollama.

In [1]:
from openai import OpenAI

# Connect to the locally running Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"   # Required by the SDK but ignored by Ollama
)

print("✅ Successfully connected to Ollama")

✅ Successfully connected to Ollama


In [ ]:

import time

start_time = time.time()

response = client.chat.completions.create(
    model="qwen3:1.7b",
    messages=[
        {
            "role": "user",
            "content": "In one sentence, explain what a Large Language Model is."
        }
    ]
)

end_time = time.time()

print("Response:")
print(response.choices[0].message.content)

print(f"\nInference Time: {end_time - start_time:.2f} seconds")

Response:
A Large Language Model is an AI model trained on massive text datasets to understand and generate human-like text, leveraging neural networks to learn patterns and context.

Inference Time: 19.53 seconds


## Observation

- Model Used: qwen3:1.7b
- Inference Time: 19.53 seconds
- The model successfully explained the concept of an LLM in a concise and grammatically correct sentence.
- Running locally through Ollama eliminates the need for an external API key or internet connectivity.

## Key Takeaways

- Ollama provides an OpenAI-compatible API.
- The OpenAI Python SDK can communicate with a locally hosted model.
- Smaller models significantly improve response latency on resource-constrained hardware.

# How Large Language Models Work

Large Language Models (LLMs) are autoregressive neural networks trained to predict the next token in a sequence. During training, the model is exposed to billions or trillions of tokens from diverse text sources. Rather than memorizing answers, it learns statistical patterns and semantic relationships between tokens.

When a user provides a prompt, the model:

1. Tokenizes the input text.
2. Converts tokens into numerical embeddings.
3. Processes the embeddings through multiple Transformer layers using the self-attention mechanism.
4. Predicts the probability distribution of the next token.
5. Selects one token based on the decoding strategy.
6. Appends the selected token to the input sequence.
7. Repeats the process until a stopping condition is reached.

This iterative next-token prediction process enables the model to generate coherent paragraphs, answer questions, write code, summarize documents, and perform many other language tasks.

## LLM Inference Pipeline

The following pipeline summarizes how an LLM processes a user query to generate a response.

```text
                User Prompt
                     │
                     ▼
            Tokenization
                     │
                     ▼
         Token IDs (Integers)
                     │
                     ▼
              Embeddings
                     │
                     ▼
      Transformer Layers
 (Self-Attention + Feed Forward)
                     │
                     ▼
      Next Token Probability
                     │
                     ▼
          Token Selection
                     │
                     ▼
        Append New Token
                     │
                     ▼
     Stop?
      │            │
     No            Yes
      │             │
      └─────────────┘
            ▼
      Final Response
```

Each iteration predicts **only one token**. The newly generated token is appended to the existing sequence, and the entire sequence is processed again until a stopping condition is reached.

> **Think:**  
> If an LLM generates only **one token at a time**, how is it able to produce coherent paragraphs, write code, or answer complex questions?
>
> *We will answer this question as we learn about tokens, embeddings, self-attention, and Transformer architectures.*

# What is a Token?

A **token** is the basic unit of text processed by a Large Language Model (LLM). Before the model can understand or generate text, the input must first be broken down into tokens through a process called **tokenization**.

A token is **not necessarily a complete word**. Depending on the tokenizer, a token may represent:

- A complete word
- Part of a word (subword)
- A punctuation mark
- A number
- A special symbol
- A whitespace character

For example, the sentence:

```
Artificial Intelligence is transforming robotics.
```

may be tokenized as:

```
["Artificial", " Intelligence", " is", " transforming", " robotics", "."]
```

or, with a different tokenizer,

```
["Art", "ificial", " Intelligence", " is", " transform", "ing", " robotics", "."]
```

Different LLMs use different tokenizers, so the same sentence may produce different token sequences.

In [ ]:
### Experiment 1 – Encoding Text into Token IDs

import tiktoken

# Load the tokenizer used by GPT-4o
encoding = tiktoken.encoding_for_model("gpt-4o")

text = "Large Language Models are transforming robotics."

tokens = encoding.encode(text)

print("Original Text:")
print(text)

print("\nToken IDs:")
print(tokens)

print("\nNumber of Tokens:")
print(len(tokens))

Original Text:
Large Language Models are transforming robotics.

Token IDs:
[42565, 20333, 50258, 553, 64779, 122246, 13]

Number of Tokens:
7


In [ ]:
### Experiment 2 – Decoding Individual Tokens

print("Token Breakdown:\n")

for token in tokens:
    decoded = encoding.decode([token])
    print(f"{token:<8} --> {repr(decoded)}")

Token Breakdown:

42565    --> 'Large'
20333    --> ' Language'
50258    --> ' Models'
553      --> ' are'
64779    --> ' transforming'
122246   --> ' robotics'
13       --> '.'


In [ ]:
### Experiment 3 – Token Count Analysis

words = [
    "robot",
    "robotics",
    "robotization",
    "Artificial",
    "Artificial Intelligence",
    "AI",
    "ChatGPT"
]

print(f"{'Word':<30} {'Tokens'}")
print("-" * 45)

for word in words:
    token_ids = encoding.encode(word)
    print(f"{word:<30} {len(token_ids)}")

Word                           Tokens
---------------------------------------------
robot                          1
robotics                       2
robotization                   2
Artificial                     1
Artificial Intelligence        2
AI                             1
ChatGPT                        2


In [ ]:
### Experiment 4 – Visualizing Token Splits
words = [
    "robot",
    "robotics",
    "robotization",
    "Artificial",
    "Artificial Intelligence",
    "AI",
    "ChatGPT"
]

for word in words:
    token_ids = encoding.encode(word)
    token_text = [encoding.decode([t]) for t in token_ids]

    print(f"\nWord: {word}")
    print(f"Token IDs : {token_ids}")
    print(f"Tokens    : {token_text}")


Word: robot
Token IDs : [33218]
Tokens    : ['robot']

Word: robotics
Token IDs : [33218, 1541]
Tokens    : ['robot', 'ics']

Word: robotization
Token IDs : [33218, 2860]
Tokens    : ['robot', 'ization']

Word: Artificial
Token IDs : [186671]
Tokens    : ['Artificial']

Word: Artificial Intelligence
Token IDs : [186671, 42378]
Tokens    : ['Artificial', ' Intelligence']

Word: AI
Token IDs : [17527]
Tokens    : ['AI']

Word: ChatGPT
Token IDs : [14065, 162016]
Tokens    : ['Chat', 'GPT']
